# Caries Risk Prediction ML Pipeline

## ⚠️ Data Governance & Security Guidelines

### Patient Privacy & HIPAA Compliance
This repository handles **sensitive clinical data** and is subject to strict data privacy protocols. To prevent accidental disclosure of Patient Identifiable Information (PII) or clinical markers:

- **Never stage or commit raw clinical datasets.** Real patient files must reside strictly in `data/`, which is enforced via `.gitignore`.
- **Use synthetic data for local development.** Only mock data in `data/mock_data/` is tracked for pipeline testing.

### Automated Notebook Output Stripping
**Cell outputs must never be committed.** Executed cells and plots can leak intermediate patient statistics or records into Git history.

Repository security is enforced via automated pre-commit hooks (`nbstripout`). Run these setup commands once upon cloning to activate local safeguards:

```bash
pip install -r requirements.txt
nbstripout --install
pre-commit install

## Import Statements

In [ ]:
# Standard Library Imports
import os
from collections import Counter
from pathlib import Path

# Third-Party Numerical & Data Handling Imports
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Machine Learning Imports (Scikit-Learn)nb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Gradient Boosting Imports
from xgboost import XGBClassifier

## Export Paths

In [ ]:
# Get the project root directory (one level up from the 'notebooks' folder)
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Define absolute paths for output directories
REPORTS_DIR = ROOT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
TABLES_DIR = REPORTS_DIR / 'tables'
MODELS_DIR = ROOT_DIR / 'models'

# Ensure directories exist at the root level
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

## Load Data

In [ ]:
df = pd.read_csv(ROOT_DIR / 'data/mock_data/mock_data.csv') # MOCK DATA

## Data Cleanup


VERY IMPORTANT: Make sure you do REMOVE any "leaky features". That is, variables that would leak the outcome you're trying to predict. This can include:
- DMFT
- Number of new cavities this year → directly tells the model the outcome

ALSO MAKE SURE TO DELETE PATIENT IDENTIFIERS:
- patient_code

In [ ]:
# 1. Drop rows where target cannot be computed
df = df.dropna(subset=['dmft_d', 'dmft_m', 'dmft_f', 'dmft_total'])

# 2. Define columns to ignore
target_col = 'caries_risk'
leaky_cols = ['dmft_d', 'dmft_m', 'dmft_f', 'dmft_total']
ignore_cols = ['patient_code', target_col] + leaky_cols

# 3. Separate X and y
X = df.drop(columns=ignore_cols)
y = df[target_col]

## Data Exploration

In [ ]:
# Check outputs
print(X.head())
print(y.value_counts())

## Train Test Split

We will do a 70-30 split.

**IMPORTANT: DO NOT RUN TRAIN-TEST-SPLIT MORE THAN ONCE** to avoid data leakage!!

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.3, 
                                                    random_state=42)

## Setup Pipelines

Instantiate the pipelines for the three models. For caries risk prediction, we will be using **classification models** as we will be predicting categorical labels (low/high caries risk)

**1. Logistic Regression**

We use a logistic regression model as a baseline for our other models: Random Forest and XGBoost. Logistic regression models are used to predict binary probabilities (low/high caries risk) by modeling the log-odds of the outcome as a linear combination of input features.

**2. Random Forest**

Random Forest is an ensemble learning method that constructs a collection of independent decision trees during training. By combining the predictions of multiple trees (via majority voting), it reduces model variance, prevents overfitting, and effectively handles non-linear relationships and interactions among clinical risk factors.

**3. XGBoost**

eXtreme Gradient Boosting (XGBoost) is an optimized gradient-boosted decision tree algorithm that builds trees sequentially, where each new tree corrects the residual errors of the previous ones. It is highly efficient for tabular clinical data, offering built-in regularization to prevent overfitting and strong predictive accuracy for complex, high-dimensional datasets.

### Handling Missing Values

Missing values will be handled through **median imputation** applied to the training set and subsequently to the test set to ensure that no information from the test set influences the imputation process, preventing data leakage. Dropping entire rows for a few missing values can result in massive data loss. During data collection, human error can occur. One or two missing values does not mean that the rest of the patient's data is useless.

Median imputation is performed as clinical data is often skewed. Clinical features such as DMFT scores, number of restorations, active lesions, etc. contain extreme values or non-normal distributions, which makes the mean unstable. A few patients with very high values can pull the mean upward artificially. Median is more robust to such outliers and presents the middle value regardless of extremes. 

**Note on XGBoost**: XGBoost handles missing values natively. However, median imputation is applied uniformly across all three models to maintain pipeline consistency and ensure compatibility with Logistic Regression and Random Forest, which require complete data.

### Handling Class Imbalances

Depending on the clinical setting and patient population, the dataset may skew towards either high or low caries risk. Handling class imbalance is therefore essential.

There are several balancing strategies available: Class Weight, Synthetic Minority Over-sampling Technique (SMOTE), Random Over Sampling, and Random Under Sampling. In our case, we will be utilizing **Class Weight** for its simplicity: Class weighting was selected as it does not generate synthetic samples, preserving the integrity of the original dataset. It applies higher penalties for misclassifying minority classes, and is natively supported by the three algorithms used in this study.

In [ ]:
class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

pipelines = {
    'log_reg': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()), 
        ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ]),
    'rf': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ]),
    'xgb': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos_weight))
    ])
}

In [ ]:
param_grids = {
    'log_reg': {
        'model__C': [0.01, 0.1, 1, 10]
    },
    'rf': {
        'model__n_estimators': [100, 300, 500],
        'model__max_depth': [None, 5, 10, 20],
        'model__min_samples_leaf': [1, 5, 10]
    },
    'xgb': {
        'model__n_estimators': [100, 300, 500],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__min_child_weight': [1, 5, 10]
    }
}

CV_FOLDS = 5 # 5 is already the default; specified here to be explicit
SCORING = 'roc_auc'


grids = {
    'log_reg': GridSearchCV(pipelines['log_reg'], param_grids['log_reg'],cv=CV_FOLDS, scoring=SCORING),
    'rf': GridSearchCV(pipelines['rf'], param_grids['rf'], cv=CV_FOLDS, scoring=SCORING),
    'xgb': GridSearchCV(pipelines['xgb'], param_grids['xgb'], cv=CV_FOLDS, scoring=SCORING)
}

## Training

In [ ]:
for name, grid in grids.items():
    grid.fit(X_train, y_train)
    print(f"{name} trained successfully.")

## EVALUATION
Evaluation performance of each model according to the following metrics:
- Accuracy
- Sensitivity
- Specificity
- Precision
- AUC-ROC

In [ ]:
performance_results = {}

for name, grid in grids.items():
    y_pred = grid.predict(X_test)
    y_proba = grid.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision = precision_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    best_est = grid.best_estimator_
    if hasattr(best_est, 'named_steps'):
        # It's a Pipeline — get the model step
        model_step = best_est.named_steps['model']
        if hasattr(model_step, 'feature_importances_'):
            importances = model_step.feature_importances_
        elif hasattr(model_step, 'coef_'):
            importances = model_step.coef_[0]
        else:
            importances = None
    else:
        # Direct model
        if hasattr(best_est, 'feature_importances_'):
            importances = best_est.feature_importances_
        elif hasattr(best_est, 'coef_'):
            importances = best_est.coef_[0]
        else:
            importances = None

    performance_results[name] = {
        'accuracy': acc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'f1_score': f1,
        'AUC-ROC': roc_auc,
        'y_proba': y_proba,
        'importances': importances
    }

    print(f"--- {name.upper()} ---")
    print(f"Best Params: {grid.best_params_}")
    print(f"Best CV Score (ROC-AUC): {grid.best_score_:.4f}")
    print(f"Test Accuracy: {acc:.4f} | Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f} | ROC-AUC: {roc_auc:.4f}\n")

## Compare Models

Now we compare the three models (Logistic Regression, Random Forest, XGBoost) based on their scores

In [ ]:
# Create summary DataFrame from performance_results
summary_df = pd.DataFrame(performance_results).T[
    ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1_score', 'AUC-ROC']
]
summary_df.columns = ['Accuracy', 'Sensitivity (Recall)', 'Specificity', 'Precision', 'F1-Score', 'ROC-AUC']

# Identify the top model for each metric dynamically
print("=== BEST PERFORMING MODEL PER METRIC ===")
for metric in summary_df.columns:
    top_model = summary_df[metric].idxmax()
    top_score = summary_df.loc[top_model, metric]
    print(f"Best {metric:12s}: {top_model.upper()} ({top_score:.4f})")

# Save CSV table to reports/tables/
summary_df.to_csv(TABLES_DIR / 'model_performance_summary.csv')
print("Saved summary table to reports/tables/model_performance_summary.csv")

summary_df

## Select Best Overall Model
To select the best overall model, we will use **ROC-AUC as the primary metric** and **sensitivity as the secondary metric**.


In [ ]:
# Sort by primary metric (ROC-AUC), then secondary metric (Sensitivity) descending
sorted_summary = summary_df.sort_values(
    by=['ROC-AUC', 'Sensitivity (Recall)'], 
    ascending=[False, False]
)

best_overall_name = sorted_summary.index[0]
best_auc = sorted_summary.loc[best_overall_name, 'ROC-AUC']
best_sens = sorted_summary.loc[best_overall_name, 'Sensitivity (Recall)']

print(f"🏆 OVERALL BEST MODEL: {best_overall_name.upper()}")
print(f"Primary Metric (ROC-AUC): {best_auc:.4f}")
print(f"Tie-Breaker (Sensitivity): {best_sens:.4f}")

### Figures

ROC Curves

In [ ]:
for name, data in performance_results.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_proba'])
    
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"{name.upper()} (AUC = {data['AUC-ROC']:.3f})", color='blue', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {name.upper()}')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

In [ ]:
# Combined ROC Plot and Export
plt.figure(figsize=(8, 6))

for name, data in performance_results.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_proba'])
    
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"{name.upper()} (AUC = {data['AUC-ROC']:.3f})", color='blue', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {name.upper()}')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Save individual ROC curve
    plt.savefig(FIGURES_DIR / f'roc_curve_{name}.png', dpi=300, bbox_inches='tight')
    plt.show()

Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
feature_names = X.columns.tolist()

for ax, (name, data) in zip(axes, performance_results.items()):
    importances = data['importances']
    
    if importances is None:
        ax.set_title(f'{name.upper()} - No Importance Available')
        continue
    
    # Get absolute values for LR coefficients
    abs_importances = np.abs(importances)
    
    # Sort by importance
    indices = np.argsort(abs_importances)[::-1]
    sorted_features = [feature_names[i] for i in indices]
    sorted_importances = abs_importances[indices]
    
    # Plot top 10
    ax.barh(sorted_features[:10][::-1], sorted_importances[:10][::-1])
    ax.set_title(f'{name.upper()} - Feature Importance')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
for name, data in performance_results.items():
    if data['importances'] is not None:
        importance_df = pd.DataFrame({
            'feature': X.columns.tolist(),
            'importance': np.abs(data['importances'])
        }).sort_values('importance', ascending=False)
        
        importance_df.to_csv(TABLES_DIR / f'feature_importance_{name}.csv', index=False)

## Export Best Model

Export the best performing caries risk prediction model for production

In [ ]:
joblib.dump(grids[best_model_name].best_estimator_, MODELS_DIR / 'caries-risk-predictor.pkl')